In [ ]:
import os, eda_analysis

cfg = eda_analysis.EdaConfig(family="arms/outcomes")
S   = eda_analysis.notebook_setup(cfg)

# `arms/outcomes` -- what every arm scored

**The question.** For each arm on disk, at each model state, on each instrument: what did the
grader give it, how did that move across iterations, and where did it end up relative to the
untrained base policy?

**The axis.** `iteration` here is the MODEL STATE (`model_iter_<N>`), which is labelled by the
policy that GENERATED those conversations -- so `iteration = 0` is the untrained base and `N`
training iterations produce `N + 1` states. It is *not* the training-iteration index; the two
are off by one.

**The pairing unit.** Every score is one conversation with one of the 96 fixed personas, and a
persona is the same client in every arm and every iteration. Nothing on this page is paired
(these are descriptives); the paired contrasts live in `lookahead/reward` and `method/contrast`.

**The graders.** Every table and figure names its grader. The default judge is the same local
model that served the training oracle, so it is *not* held out; a second judge, if the lake holds
one, is. They are reported side by side and **never averaged** -- one is train and one is test,
they do not share a scale, and a mean over them applies a silent model-dependent shrinkage.

**Orientation.** Eight instruments are higher-is-better. **MICI is lower-is-better** -- it counts
MI-INCONSISTENT therapist behaviour -- so every ranking below is taken over
`sign_of(metric) * score` and every "gain" column is signed so that **positive always means
better**, on every metric.

**Final AND best.** Both endpoints are reported for every arm. An arm that peaked and then
regressed is flattered by the best-state row and not by the final-state row, and quoting only one
of them chooses the answer; `past_peak = True` marks exactly the arms where the two disagree.

**Settings.** Arms that differ in more than the two levers (method, K) -- a different therapist,
oracle or patient model, rubric, MCL or branch width -- belong to different *settings*
(`data.setting_tags`), and each setting starts from its own untrained policy. So every figure
and the by-iteration table show ONE setting -- the Instruct arms and the base-model arms are never
drawn in the same panel -- and the leaderboard ranks arms WITHIN their setting: one ranking across
a base-model therapist and an Instruct one would mostly rank their starting points. The default
setting's artifacts keep their plain names; any other setting's carry its tag
(`trajectory_gpt4m_ThL1B`).

**What this family does not claim.** Overlapping CI bands are not a test. The bands are bootstrap
intervals ACROSS the 96 personas, and persona variance dominates -- a paired contrast removes it,
so two heavily overlapping bands are perfectly compatible with a decisive within-persona
difference. Read differences off the contrast families, never off these panels.

In [ ]:
# ---------------------------------------------------------------------------
# Imports, both graders, and the guards that let this notebook render with NO DATA
# on disk -- which is the normal state until the first arm has been generated and
# scored. Every section below degrades to an explicit "no data yet" artifact.
# ---------------------------------------------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from eda_analysis import constants, data, exports, plotting, stats

SEED = S.CFG.boot_seed
FOCUS = S.CFG.focus_metric

# Every grader in the lake, in ONE frame with a `judge` column. Never averaged across
# judges; always shown side by side.
ALL = data.scores_by_judge(S.ARMS, rep=S.CFG.judge_rep, attach_persona=S.CFG.attach_persona)
if ALL.empty and not S.SCORES.empty:
    ALL = S.SCORES
HAVE = not ALL.empty

JUDGES = sorted(ALL["judge"].unique()) if HAVE else [S.JUDGE]
METRICS = [m for m in constants.METRIC_ORDER if HAVE and m in set(ALL["metric"].unique())]
ARMS = sorted(ALL["arm_label"].unique()) if HAVE else []
PALETTE = plotting.arm_palette(ARMS) if ARMS else {}
# experiment_name -> setting tag ("" = the default setting), and the same keyed on the display
# label. A setting is everything but the two levers (method, K); each starts from its own
# untrained policy, so reference levels and ranks are taken per setting, never pooled across them.
SETTING = data.setting_tags(S.ARMS)
SETTING_BY_LABEL = {a.label: SETTING[a.experiment_name] for a in S.ARMS}
# setting tag -> its EXPLICIT name for titles ("ThL1Bi" / "ThL1B"; "" while only one setting is on
# disk), and the settings that have scores, default first.
NAMES = data.setting_names(S.ARMS)
SETTING_NAME = {SETTING[a.experiment_name]: NAMES[a.experiment_name] for a in S.ARMS}
SETTINGS = sorted({SETTING_BY_LABEL.get(a, "") for a in ARMS})

NO_DATA = ("NO DATA YET -- no scored conversations for these arms. Generate an arm, then run "
           "notebooks/scoring/Run_Eval.ipynb; this family re-renders from the score lake.")

# A clean leaf per render: reset_results clears ONLY this family's figures/ + tables/
# (the hand-authored PRESERVE names are out of reach structurally), and provenance is
# then re-written from the frame actually loaded -- both graders, not just the default.
exports.reset_results()
exports.save_provenance(S.CFG, ALL)


def placeholder(name, message=NO_DATA, group=None, caption=None):
    # A figure-shaped marker, so a section that had nothing to plot still appears in the
    # index saying so, instead of leaving a hole a reader has to interpret.
    fig = plt.figure(figsize=(7.6, 1.9))
    ax = fig.add_subplot(111)
    ax.axis("off")
    ax.text(0.5, 0.5, message, ha="center", va="center", fontsize=8.5,
            color="#777777", wrap=True)
    exports.save_fig(fig, name, group=group, caption=caption or message)
    plt.close(fig)


def base_levels(frame):
    # The untrained level of each SETTING: the mean over that setting's arms at model state 0.
    # One line pooled across settings would sit between two starting points and describe neither.
    base = frame[frame["iteration"] == 0]
    if base.empty:
        return {}
    tags = base["experiment_name"].map(SETTING).fillna("")
    return {tag: float(g["score"].mean()) for tag, g in base.groupby(tags, sort=True)}


def draw_base_lines(ax, levels):
    # One dotted line per setting, named after it ("base", "base, ThL1B").
    for tag, level in levels.items():
        plotting.add_base_line(ax, level, label="base" + (f", {tag}" if tag else ""))


def setting_arms(labels, setting):
    # The arm labels of ONE setting, sorted: the series a per-setting figure or table shows.
    return sorted(a for a in labels if SETTING_BY_LABEL.get(a, "") == setting)


def name_tail(setting):
    # Artifact-name suffix: none for the default setting (its names never move), "_<tag>" else.
    return f"_{setting}" if setting else ""


def titled(text, setting):
    # Name the setting in a title whenever more than one is on disk ("... -- ThL1B").
    name = SETTING_NAME.get(setting, "")
    return f"{text} -- {name}" if name else text


print(f"judges : {JUDGES}")
print(f"arms   : {ARMS or '(none on disk)'}")
print(f"settings: {[SETTING_NAME.get(s) or s or 'default' for s in SETTINGS] or '(none scored)'}")
print(f"metrics: {METRICS or '(none scored)'}")
print(f"focus  : {FOCUS}   seed: {SEED}")
if not HAVE:
    print(NO_DATA)

## 1. Coverage -- what is actually scored

Before any number is read, this says which (grader, arm, model state) cells exist and how much of
the 96-persona grid each one covers. A partly-scored state is not an error -- it is the normal
condition between a training run and a scoring run -- but it silently shortens every contrast
computed from it, on both sides, so it belongs next to the numbers rather than in a log.

In [ ]:
def coverage(df):
    # One row per (grader, arm, model state): how many personas and instruments landed.
    if df.empty:
        return pd.DataFrame()
    keys = ["judge", "arm_label", "iteration"]
    out = (df.groupby(keys, as_index=False)
             .agg(model_state=("model_state", "first"),
                  metrics=("metric", "nunique"),
                  personas=("persona_id", "nunique"),
                  rows=("score", "size"),
                  scored_cells=("score", "count")))
    out["ungraded_cells"] = out["rows"] - out["scored_cells"]
    # Completeness is PER INSTRUMENT. Pooling the metric dimension hides exactly the failure
    # this column exists to catch: one instrument missing a third of its personas still shows
    # 96 distinct persona_ids, because the other instruments supply them -- while every
    # contrast on that instrument quietly runs at the smaller n. `notna` too: an ungraded cell
    # is a present row with no score, and it shortens a contrast just as a missing row does.
    graded = df[df["score"].notna()]
    worst = (graded.groupby(keys + ["metric"], as_index=False)
                   .agg(n=("persona_id", "nunique"))
                   .groupby(keys, as_index=False)
                   .agg(min_personas_per_metric=("n", "min")))
    out = out.merge(worst, on=keys, how="left")
    out["min_personas_per_metric"] = out["min_personas_per_metric"].fillna(0).astype(int)
    # A state that is missing a whole instrument has no row to be the minimum of, so the
    # instrument count is checked against the instruments present anywhere in this frame.
    n_instruments = int(df["metric"].nunique())
    out["complete_grid"] = ((out["min_personas_per_metric"] == constants.N_PERSONAS)
                            & (out["metrics"] == n_instruments))
    return (out.drop(columns=["rows"])
               .rename(columns={"arm_label": "arm"})
               .sort_values(["judge", "arm", "iteration"])
               .reset_index(drop=True))


COV = coverage(ALL)
exports.save_table(
    COV, "coverage",
    caption=("Scored coverage per grader x arm x MODEL STATE (iteration 0 = untrained base). "
             "`personas` pools every instrument, so read `min_personas_per_metric` instead: it "
             "is the worst-covered instrument's count of distinct persona_ids with a non-null "
             "score, and it is what `complete_grid` is built from (together with the instrument "
             "count, so a wholly missing rubric also shows). Anything below 96 means the state "
             "is only partly scored on at least one instrument, and every paired contrast on it "
             "drops those personas from BOTH sides. `ungraded_cells` are rows the grader "
             "returned nothing for. `metrics` counts instruments, not items."))
if COV.empty:
    print(NO_DATA)
else:
    partial = COV[~COV["complete_grid"]]
    print(f"{len(COV)} (grader, arm, state) cells; {len(partial)} below the full 96-persona "
          f"grid on at least one instrument")
    display(COV.head(20))

## 2. Descriptives -- mean, spread and a seeded bootstrap CI

One row per (grader, arm, model state, instrument). `ci_lo`/`ci_hi` are a 2,000-resample
percentile bootstrap of the MEAN **across personas**, seeded with `BOOT_SEED` so that
re-rendering unchanged data reproduces the table byte for byte.

The interval describes the spread across the 96 clients. It is not the uncertainty of a
difference between two arms -- that is a paired quantity, and pairing removes exactly the persona
variance these intervals are made of.

In [ ]:
def describe(df):
    # Per (grader, arm, state, metric): n, mean, sd and a seeded bootstrap CI of the mean.
    if df.empty:
        return pd.DataFrame()
    rows = []
    for (judge, arm, state, metric), g in df.groupby(
            ["judge", "arm_label", "iteration", "metric"], sort=True):
        x = pd.to_numeric(g["score"], errors="coerce").to_numpy(dtype=float)
        x = x[~np.isnan(x)]
        lo, hi = stats.bootstrap_ci(x, np.mean, seed=SEED) if x.size else (np.nan, np.nan)
        rows.append({
            "judge": judge, "arm": arm, "iteration": int(state), "metric": metric,
            "instrument": constants.short_label(metric),
            "n": int(x.size),
            "mean": float(np.mean(x)) if x.size else np.nan,
            "sd": float(np.std(x, ddof=1)) if x.size > 1 else np.nan,
            "ci_lo": lo, "ci_hi": hi,
            "sign": int(constants.sign_of(metric)),
        })
    out = pd.DataFrame(rows)
    return out.sort_values(["judge", "metric", "arm", "iteration"]).reset_index(drop=True)


DESC = describe(ALL)
exports.save_table(
    DESC, "descriptives",
    caption=("Per grader x arm x MODEL STATE x instrument: n personas, mean, SD, and a 2,000-"
             "resample percentile bootstrap CI of the mean ACROSS personas (seed=BOOT_SEED). "
             "`sign` is +1 where higher is better and -1 for MICI. These CIs are unpaired: do "
             "not read an arm difference off two of them."))
if DESC.empty:
    print(NO_DATA)
else:
    display(DESC.head(20))

## 2b. By iteration -- every look-ahead arm at every iteration, one table per setting

The table to scan: one row per (instrument, iteration), one column pair per arm of ONE setting --
the Instruct arms and the base-model arms never share a table -- holding that arm's mean and its
95% CI across the 96 personas. Iteration 0 is the untrained policy; iteration N is the policy after
N training iterations (`model_iter_N`). These are the descriptives above, reshaped: no arm is
subtracted from another here (the paired contrasts are `lookahead/reward`'s and
`method/contrast`'s).

`by_iteration_<grader>` is the default setting; any other carries its tag
(`by_iteration_<grader>_ThL1B`).

In [ ]:
def by_iteration(desc, judge, arms):
    # One row per (instrument, iteration); per arm, its mean and its 95% CI across personas.
    sub = desc[(desc["judge"] == judge) & desc["arm"].isin(arms)] if not desc.empty else desc
    if sub.empty:
        return pd.DataFrame()
    order = {m: i for i, m in enumerate(constants.METRIC_ORDER)}
    rows = []
    for (metric, state), g in sub.groupby(["metric", "iteration"], sort=False):
        row = {"_order": order.get(metric, len(order)),
               "instrument": constants.short_label(metric), "iteration": int(state)}
        cells = g.set_index("arm")
        for arm in arms:
            if arm in cells.index:
                r = cells.loc[arm]
                row[arm] = float(r["mean"])
                row[f"{arm} 95% CI"] = f"[{float(r['ci_lo']):.3f}, {float(r['ci_hi']):.3f}]"
            else:
                row[arm] = np.nan
                row[f"{arm} 95% CI"] = ""
        rows.append(row)
    out = pd.DataFrame(rows).sort_values(["_order", "iteration"], kind="mergesort")
    return out.drop(columns="_order").reset_index(drop=True)


BY_ITER = {}
for judge in JUDGES:
    for setting in SETTINGS or [""]:
        own = setting_arms(ARMS, setting)
        table = by_iteration(DESC, judge, own)
        BY_ITER[(judge, setting)] = table
        where = f", setting {SETTING_NAME[setting]}" if SETTING_NAME.get(setting) else ""
        exports.save_table(
            table, f"by_iteration_{judge}{name_tail(setting)}",
            caption=(f"Every arm of ONE setting at every iteration, grader {judge}{where}: one row "
                     f"per (instrument, iteration), one mean + 95% CI column pair per arm "
                     f"({', '.join(own) or 'none scored'}). Iteration 0 is the untrained policy, "
                     f"N the policy after N training iterations. CIs are 2,000-resample bootstraps "
                     f"of the mean ACROSS the 96 personas (seed=BOOT_SEED) -- spread, not the "
                     f"precision of an arm difference. MICI is lower-is-better."))
        if not table.empty:
            print(f"{judge} / {SETTING_NAME.get(setting) or 'default setting'}:")
            display(table[table["instrument"] == constants.short_label(FOCUS)])
if not BY_ITER:
    print(NO_DATA)

## 3. Trajectories -- one figure per setting, one panel per instrument

One figure per grader AND setting -- the Instruct arms and the base-model arms are never drawn in
the same panel -- one panel per instrument, one line per arm, in its stable colour (an arm is the
same colour in every artifact of this EDA). The default setting keeps the plain name
(`trajectory_<grader>`); any other carries its tag (`trajectory_<grader>_ThL1B`) and names itself
in the title. The dotted reference is that setting's **base level**: the mean over `model_iter_0`
of its arms, i.e. its untrained policy on that instrument under that grader.

The band is the same unpaired bootstrap CI as the table above -- spread across personas, not the
precision of an arm difference.

In [ ]:
for judge in JUDGES:
    for setting in SETTINGS or [""]:
        name = f"trajectory_{judge}{name_tail(setting)}"
        sub = (ALL[(ALL["judge"] == judge) & ALL["arm_label"].isin(setting_arms(ARMS, setting))]
               if HAVE else ALL)
        if sub.empty or not METRICS:
            placeholder(name, caption=f"{NO_DATA} (grader {judge})")
            continue
        fig, axes = plotting.grid(len(METRICS), ncols=3)
        for ax, m in zip(axes, METRICS):
            panel = sub[sub["metric"] == m]
            # ONE setting per figure, so one base level: its arms' model_iter_0 conversations.
            levels = base_levels(panel)
            plotting.score_trajectory(
                panel, metric=m, metric_col="metric", arm_col="arm_label", palette=PALETTE,
                base_value=next(iter(levels.values()), None), ax=ax,
                title=constants.short_label(m), xlabel="model state", ylabel="grader score")
            legend = ax.get_legend()
            if legend is not None and ax is not axes[0]:
                legend.remove()
        if SETTING_NAME.get(setting):
            fig.suptitle(titled(f"grader {judge}", setting), y=1.01, fontweight="bold")
        where = f", setting {SETTING_NAME[setting]}" if SETTING_NAME.get(setting) else ""
        exports.save_fig(
            fig, name,
            caption=(f"Mean score by MODEL STATE, one panel per instrument, grader {judge}{where}: "
                     f"the arms of ONE setting only. State 0 is the untrained base; the dotted "
                     f"line is this setting's base level, pooled over its arms. Bands are "
                     f"unpaired 95% bootstrap CIs across the 96 personas (seed=BOOT_SEED) -- "
                     f"overlap between two bands is NOT evidence of no difference. MICI is "
                     f"lower-is-better."))
        plt.close(fig)
        print(f"trajectory rendered for grader {judge}{where}")

## 4. Leaderboard -- final state and best state, side by side

For every (grader, instrument, arm):

* `final_*` is the arm's **last** trained model state;
* `best_*` is the state maximising `sign_of(metric) * mean`, i.e. the best checkpoint *on that
  instrument for that grader*;
* `gain_final` / `gain_best` are signed against the base so that **positive is always better**,
  MICI included;
* `past_peak = True` means the best state is not the final state -- the arm regressed after its
  peak, and the two endpoints tell different stories.

Ranks are computed on the oriented value, so MICI ranks the right way round without a special
case at the call site -- and WITHIN a setting (`data.setting_tags`): arms that differ in more than
method and K start from different untrained policies, so each setting has its own rank 1.

In [ ]:
def leaderboard(desc):
    # Final-state and best-state endpoints per (grader, instrument, arm), both signed as gains.
    if desc.empty:
        return pd.DataFrame()
    rows = []
    for (judge, metric, arm), g in desc.groupby(["judge", "metric", "arm"], sort=True):
        g = g.sort_values("iteration")
        sign = int(constants.sign_of(metric))
        base = g[g["iteration"] == 0]["mean"]
        base_mean = float(base.iloc[0]) if len(base) else np.nan
        trained = g[g["iteration"] > 0]
        if trained.empty:
            continue
        oriented = sign * trained["mean"].to_numpy(dtype=float)
        if np.all(np.isnan(oriented)):
            continue
        final = trained.iloc[-1]
        best = trained.iloc[int(np.nanargmax(oriented))]
        rows.append({
            "judge": judge, "metric": metric, "instrument": constants.short_label(metric),
            "arm": arm, "sign": sign, "n_states": int(len(trained)),
            "base_mean": base_mean,
            "final_state": int(final["iteration"]), "final_mean": float(final["mean"]),
            "best_state": int(best["iteration"]), "best_mean": float(best["mean"]),
            "gain_final": sign * (float(final["mean"]) - base_mean),
            "gain_best": sign * (float(best["mean"]) - base_mean),
            "past_peak": int(best["iteration"]) != int(final["iteration"]),
        })
    out = pd.DataFrame(rows)
    if out.empty:
        return out
    out["oriented_final"] = out["sign"] * out["final_mean"]
    out["oriented_best"] = out["sign"] * out["best_mean"]
    # Ranked WITHIN a setting: a base-model therapist and an Instruct one start from different
    # untrained policies, so one ranking across them would mostly rank the two starting points.
    out["setting"] = out["arm"].map(SETTING_BY_LABEL).fillna("")
    grp = out.groupby(["judge", "metric", "setting"])
    # Ranked on the ORIENTED value, so MICI ranks the right way round with no special case.
    # Int64 (nullable) rather than int: a cell whose mean is NaN has no rank, and a plain
    # astype(int) on that raises rather than leaving the gap visible.
    out["rank_final"] = grp["oriented_final"].rank(ascending=False, method="min").astype("Int64")
    out["rank_best"] = grp["oriented_best"].rank(ascending=False, method="min").astype("Int64")
    cols = ["judge", "metric", "instrument", "arm", "rank_final", "rank_best", "base_mean",
            "final_state", "final_mean", "gain_final", "best_state", "best_mean", "gain_best",
            "past_peak", "n_states", "sign"]
    return (out.sort_values(["judge", "metric", "setting", "rank_final"])[cols]
               .reset_index(drop=True))


LB = leaderboard(DESC)
exports.save_table(
    LB, "leaderboard",
    caption=("Final-state AND best-state endpoint per grader x instrument x arm. `gain_*` is "
             "signed by sign_of(metric) against the arm's own base (state 0), so POSITIVE IS "
             "BETTER on every instrument, MICI included; ranks are on the oriented mean and "
             "WITHIN a setting (arms that differ only in method and K), so each setting has its "
             "own rank 1. "
             "`past_peak` marks an arm whose best state is not its last -- reporting only one "
             "endpoint for those arms chooses the answer."))
exports.save_table(
    LB[LB["metric"] == FOCUS] if not LB.empty else LB, "leaderboard_focus",
    caption=(f"The leaderboard restricted to the training-reward axis ({FOCUS}) -- the metric "
             f"the policy was actually optimized against. Same signing and ranking rules."))
if LB.empty:
    print(NO_DATA)
else:
    peaked = LB[LB["past_peak"]]
    print(f"{len(LB)} endpoint rows; {len(peaked)} (grader, instrument, arm) cells are PAST PEAK")
    display(LB[LB["metric"] == FOCUS])

## 5. Endpoint distributions -- the spread behind each headline mean

The per-persona distribution at each arm's own **final** state and at its own **best** state, on
the training-reward axis, one figure per setting. The diamond is the mean with an unpaired bootstrap CI; the box shows the
median, which is not the statistic any table here reports, which is why both are drawn.

The same 96 personas appear in every box, so two boxes overlapping heavily is entirely compatible
with a large, consistent within-persona difference.

In [ ]:
def endpoint_frame(df, lb, judge, metric, state_col):
    # Per-persona rows at each arm's OWN endpoint state (final or best) -- arms peak at
    # different iterations, so a single shared state would not be either endpoint.
    empty = df.iloc[0:0]
    if df.empty or lb.empty:
        return empty
    sel = lb[(lb["judge"] == judge) & (lb["metric"] == metric)][["arm", state_col]]
    parts = []
    for _, r in sel.iterrows():
        parts.append(df[(df["judge"] == judge) & (df["arm_label"] == r["arm"])
                        & (df["metric"] == metric) & (df["iteration"] == int(r[state_col]))])
    return pd.concat(parts, ignore_index=True) if parts else empty


for judge, setting in [(j, s) for j in JUDGES for s in SETTINGS or [""]]:
    own = setting_arms(ARMS, setting)
    for state_col, tag in (("final_state", "final"), ("best_state", "best")):
        name = f"endpoint_{tag}_{judge}{name_tail(setting)}"
        frame = endpoint_frame(ALL, LB[LB["arm"].isin(own)] if not LB.empty else LB,
                               judge, FOCUS, state_col)
        if frame.empty:
            placeholder(name, caption=f"{NO_DATA} (grader {judge}, {tag} state, {FOCUS})")
            continue
        levels = base_levels(ALL[(ALL["judge"] == judge) & (ALL["metric"] == FOCUS)
                                 & ALL["arm_label"].isin(own)])
        # arm_distribution overlays a seaborn stripplot, whose JITTER draws from numpy's global
        # RNG and accepts no seed argument -- so without this, two renders of identical data
        # produce different PNGs and every tracked figure churns in git. Its bootstrap CI is
        # already seeded inside plotting; the jitter is the one draw that callsite cannot reach.
        np.random.seed(SEED)
        fig = plotting.arm_distribution(
            frame, metric=FOCUS, metric_col="metric", arm_col="arm_label", palette=PALETTE,
            title=titled(f"{constants.short_label(FOCUS)} at each arm's {tag} state ({judge})",
                         setting),
            ylabel="grader score")
        draw_base_lines(fig.axes[0], levels)
        exports.save_fig(
            fig, name,
            caption=(f"Per-persona {FOCUS} at each arm's OWN {tag} model state, grader {judge}"
                     f"{', setting ' + SETTING_NAME[setting] if SETTING_NAME.get(setting) else ''}"
                     f" (the arms of one setting only). Box = median, diamond = mean with an "
                     f"unpaired 95% bootstrap CI (seed=BOOT_SEED), dotted line = this setting's "
                     f"untrained base level. The same 96 personas are in every box, so overlap "
                     f"here does not bound a paired difference."))
        plt.close(fig)
print("endpoint distributions rendered")

## 6. Number ledger and index

A citable ledger of the few numbers this family is actually quoted for, each with the table it
came from, followed by the index refresh that ends every family notebook.

In [ ]:
values = {
    "coverage.judges": {"value": JUDGES, "source": "tables/coverage.md",
                        "note": "graders present in the score lake"},
    "coverage.arms": {"value": ARMS, "source": "tables/coverage.md",
                      "note": "arm display labels; key on experiment_name for data"},
    "coverage.metrics": {"value": METRICS, "source": "tables/coverage.md", "note": ""},
    "coverage.scored_rows": {"value": int(len(ALL)), "source": "tables/coverage.md",
                             "note": "one row per (arm, state, persona, instrument, grader)"},
    "focus.metric": {"value": FOCUS, "source": "", "note": "the training-reward axis"},
}
if not LB.empty:
    focus_rows = LB[LB["metric"] == FOCUS]
    # Leaders are per SETTING, like the ranks they are read off; the default setting keeps the
    # plain focus.<judge>.* keys.
    focus_setting = focus_rows["arm"].map(SETTING_BY_LABEL).fillna("")
    for judge, tag in sorted(set(zip(focus_rows["judge"], focus_setting))):
        jr = focus_rows[(focus_rows["judge"] == judge)
                        & (focus_setting == tag)].sort_values("rank_final")
        if jr.empty:
            continue
        key = f"focus.{judge}" + (f".{tag}" if tag else "")
        top_final = jr.iloc[0]
        top_best = jr.sort_values("rank_best").iloc[0]
        values[f"{key}.leader_final"] = {
            "value": str(top_final["arm"]), "source": "tables/leaderboard_focus.md",
            "note": (f"highest {FOCUS} at its FINAL state "
                     f"(state {int(top_final['final_state'])}, gain "
                     f"{float(top_final['gain_final']):.3f} vs base)")}
        values[f"{key}.leader_best"] = {
            "value": str(top_best["arm"]), "source": "tables/leaderboard_focus.md",
            "note": (f"highest {FOCUS} at its BEST state "
                     f"(state {int(top_best['best_state'])}, gain "
                     f"{float(top_best['gain_best']):.3f} vs base)")}
        values[f"{key}.past_peak_arms"] = {
            "value": sorted(jr[jr["past_peak"]]["arm"].tolist()),
            "source": "tables/leaderboard_focus.md",
            "note": "arms whose best state is not their last -- report both endpoints"}

exports.save_numbers(
    "outcomes", values,
    caption=("Citable headline numbers for this family, each with the table it was read off. "
             "Leaders are per grader and never pooled across graders."))
print(exports.build_index())